# Hafta 3 — A* Rota Algoritması (ilk basit versiyon)
**Sultan** — AteşKes (EERİS+)

Bu notebook Hafta 3 teslimini içerir: küçük bir örnek grafik (5 düğüm) üzerinde `networkx`'in A* implementasyonu, özelleştirilmiş `cost_hesapla()` maliyet fonksiyonu ve yeni eklenen `egim_cezasi()` terimi. Amaç: müdahale/tahliye ekiplerini gereksiz yere çok dik yollara yönlendirmemek.

Esma'nın gerçek yol verisi (OSM/Overpass, highway sınıflarına göre) henüz hazır olmadığı için plan gereği önce küçük, elle kurulmuş bir test grafiğiyle doğruluyoruz; gerçek yol verisi gelince aynı `cost_hesapla()` fonksiyonu değişmeden kullanılacak.

In [1]:
import networkx as nx
from hesaplamalar import cost_hesapla, egim_cezasi, haversine_mesafe

## 1. Maliyet fonksiyonu
`Cost = Risk + Yayılım Riski + Mesafe − Öncelik + Zaman + Erişilebilirlik + Eğim Cezası`

`egim_cezasi()` eşikleri: <10° ceza yok, 10-20° hafif (5), 20-30° orta (15), ≥30° yüksek (30) — çok dik bir segment, kısa mesafe avantajını fazlasıyla götürür.

In [2]:
for e in [5, 15, 25, 35]:
    print(f"egim={e}° -> ceza={egim_cezasi(e)}")

egim=5° -> ceza=0
egim=15° -> ceza=5
egim=25° -> ceza=15
egim=35° -> ceza=30


## 2. Test grafiği (5 düğüm)
Gerçek Hafta 1 verisinden yangın başlangıç noktası (Gürece — köy) ve tahliye hedefi (Bodrum Amerikan Hastanesi) kullanıyoruz; aralarına iki alternatif rota kuran sentetik kavşaklar (`kavsak_dik`, `kavsak_duz`, `kavsak_orta`) ekliyoruz:
- **Dik rota** (kısa, ~2.7 km): son bacakta eğim 32°
- **Düz rota** (uzun, ~5.4 km): her bacakta eğim ≤11°

In [3]:
G = nx.Graph()

G.add_node("yangin_baslangic", lat=37.039538, lon=27.322432)   # Gurece (koy)
G.add_node("kavsak_dik", lat=37.037, lon=27.365)
G.add_node("kavsak_duz", lat=37.041, lon=27.360)
G.add_node("kavsak_orta", lat=37.040, lon=27.395)
G.add_node("hastane_hedef", lat=37.039939, lon=27.428962)      # Bodrum Amerikan Hastanesi

# Kisa ama DIK rota
G.add_edge("yangin_baslangic", "kavsak_dik",
           risk=0.3, yayilma_riski=0.2, mesafe=1.5, oncelik=0.5, zaman=0.2, erisilebilirlik=0.1, egim_derece=8)
G.add_edge("kavsak_dik", "hastane_hedef",
           risk=0.3, yayilma_riski=0.2, mesafe=1.2, oncelik=0.5, zaman=0.2, erisilebilirlik=0.3, egim_derece=32)  # cok dik

# Uzun ama DUZ rota
G.add_edge("yangin_baslangic", "kavsak_duz",
           risk=0.3, yayilma_riski=0.2, mesafe=2.0, oncelik=0.5, zaman=0.2, erisilebilirlik=0.1, egim_derece=6)
G.add_edge("kavsak_duz", "kavsak_orta",
           risk=0.3, yayilma_riski=0.2, mesafe=1.8, oncelik=0.5, zaman=0.2, erisilebilirlik=0.1, egim_derece=9)
G.add_edge("kavsak_orta", "hastane_hedef",
           risk=0.3, yayilma_riski=0.2, mesafe=1.6, oncelik=0.5, zaman=0.2, erisilebilirlik=0.1, egim_derece=11)

list(G.edges(data=True))

[('yangin_baslangic',
  'kavsak_dik',
  {'risk': 0.3,
   'yayilma_riski': 0.2,
   'mesafe': 1.5,
   'oncelik': 0.5,
   'zaman': 0.2,
   'erisilebilirlik': 0.1,
   'egim_derece': 8}),
 ('yangin_baslangic',
  'kavsak_duz',
  {'risk': 0.3,
   'yayilma_riski': 0.2,
   'mesafe': 2.0,
   'oncelik': 0.5,
   'zaman': 0.2,
   'erisilebilirlik': 0.1,
   'egim_derece': 6}),
 ('kavsak_dik',
  'hastane_hedef',
  {'risk': 0.3,
   'yayilma_riski': 0.2,
   'mesafe': 1.2,
   'oncelik': 0.5,
   'zaman': 0.2,
   'erisilebilirlik': 0.3,
   'egim_derece': 32}),
 ('kavsak_duz',
  'kavsak_orta',
  {'risk': 0.3,
   'yayilma_riski': 0.2,
   'mesafe': 1.8,
   'oncelik': 0.5,
   'zaman': 0.2,
   'erisilebilirlik': 0.1,
   'egim_derece': 9}),
 ('kavsak_orta',
  'hastane_hedef',
  {'risk': 0.3,
   'yayilma_riski': 0.2,
   'mesafe': 1.6,
   'oncelik': 0.5,
   'zaman': 0.2,
   'erisilebilirlik': 0.1,
   'egim_derece': 11})]

## 3. Kenar maliyeti + heuristic
`networkx.astar_path`'e `weight` olarak `cost_hesapla()`'yı sarmalayan bir fonksiyon veriyoruz. Heuristic için düğümler arası kuş uçuşu mesafeyi (`haversine_mesafe`, km'ye çevrilmiş) kullanıyoruz — **not:** `cost_hesapla`'da `-oncelik` terimi negatif olabildiği için bu heuristic'in matematiksel olarak tam "admissible" olduğu garanti değil; bu ay için kabul edilebilir bir basitleştirme, gerçek yol ağına ölçeklenirken (Hafta 13-14) tekrar gözden geçirilecek.

In [4]:
def kenar_maliyeti(u, v, kenar_verisi):
    return cost_hesapla(
        risk=kenar_verisi["risk"],
        yayilma_riski=kenar_verisi["yayilma_riski"],
        mesafe=kenar_verisi["mesafe"],
        oncelik=kenar_verisi["oncelik"],
        zaman=kenar_verisi["zaman"],
        erisilebilirlik=kenar_verisi["erisilebilirlik"],
        egim_derece=kenar_verisi["egim_derece"],
    )


def heuristic(u, v):
    nu, nv = G.nodes[u], G.nodes[v]
    return haversine_mesafe(nu["lat"], nu["lon"], nv["lat"], nv["lon"]) / 1000  # km

## 4. Hafta 3 teslimi: A* rotayı hesapla ve karşılaştır

In [5]:
rota = nx.astar_path(G, "yangin_baslangic", "hastane_hedef", heuristic=heuristic, weight=kenar_maliyeti)
toplam_maliyet = nx.astar_path_length(G, "yangin_baslangic", "hastane_hedef", heuristic=heuristic, weight=kenar_maliyeti)

print("Secilen rota:", " -> ".join(rota))
print("Toplam maliyet:", round(toplam_maliyet, 2))

Secilen rota: yangin_baslangic -> kavsak_duz -> kavsak_orta -> hastane_hedef
Toplam maliyet: 11.3


In [6]:
dik_rota = ["yangin_baslangic", "kavsak_dik", "hastane_hedef"]
duz_rota = ["yangin_baslangic", "kavsak_duz", "kavsak_orta", "hastane_hedef"]

def rota_maliyeti(rota):
    return sum(kenar_maliyeti(u, v, G[u][v]) for u, v in zip(rota, rota[1:]))

def rota_mesafesi(rota):
    return sum(G[u][v]["mesafe"] for u, v in zip(rota, rota[1:]))

print(f"Dik rota  ({rota_mesafesi(dik_rota)} km, son bacak egim=32°): maliyet={round(rota_maliyeti(dik_rota),2)}")
print(f"Duz rota  ({rota_mesafesi(duz_rota)} km, tum bacaklar egim<=11°): maliyet={round(rota_maliyeti(duz_rota),2)}")
print()
print("A* dik (kisa) rotayi degil, guvenli (uzun) rotayi sectiyse egim cezasi calisiyor demektir." if rota == duz_rota else "UYARI: A* dik rotayi secti, egim cezasi beklendigi gibi calismiyor olabilir.")

Dik rota  (2.7 km, son bacak egim=32°): maliyet=33.5
Duz rota  (5.4 km, tum bacaklar egim<=11°): maliyet=11.3

A* dik (kisa) rotayi degil, guvenli (uzun) rotayi sectiyse egim cezasi calisiyor demektir.


## Notlar / sonraki adım
- `egim_cezasi()` ve `cost_hesapla()` artık `hesaplamalar.py`'de — Esma gerçek yol verisini (Hafta 13-14, OSM highway ağırlıkları) bu fonksiyonlarla besleyebilir.
- Heuristic (kuş uçuşu mesafe) bu ay için kabul edilebilir bir basitleştirme; gerçek yol ağına ölçeklenirken sensitivity testi (Hafta 14'te planlı) ile yeniden değerlendirilecek.
- Şu an tüm kenarlarda `risk`, `yayilma_riski`, `oncelik`, `zaman`, `erisilebilirlik` sabit tutuldu (yalnızca eğim ve mesafe değişti) — bilinçli bir izolasyon, amaç sadece eğim cezasının etkisini net görmekti. Gerçek veriyle bu değerler segment segment değişecek.